In [1]:
from vislearnlabpy.embeddings.generate_embeddings import EmbeddingGenerator, EmbeddingConfig
from vislearnlabpy.embeddings.embedding_store import EmbeddingStore
from vislearnlabpy.embeddings.utils import zscore_embeddings
import numpy as np
import pandas as pd
from vislearnlabpy.embeddings.stimuli_loader import ImageExtractor, ImgExtractionSettings, StimuliLoader
from vislearnlabpy.embeddings.similarity_utils import compute_rdm, plot_rdm

In [ ]:
clip_extraction_settings = ImgExtractionSettings(
    resize_dim=224,
    apply_content_crop=True,
    apply_center_crop=False,
    use_thumbnail=False,
    bg_threshold=1, # remove any small background artifacts that are size 1 or less
    filter_edge_artifacts=False, 
    normalize_stroke_thickness=False
)

clip_transforms = ImageExtractor.get_transformations(clip_extraction_settings)

clip_config = EmbeddingConfig(text_prompt = "a drawing of ", 
                            model_type="clip",
                            device="cuda:0",  
                            transform=clip_transforms)



clip_generator = EmbeddingGenerator(config=clip_config)

# generate embeddings for drawings  
clip_generator.generate_image_embeddings(
    #output_path="../data/embeddings/clip_sketch_embeddings", 
    input_csv="../data/clean/drawing_highres_clean.csv", 
    batch_size=100, 
    overwrite=True
)

Calculating text embeddings: 100%|██████████| 10/10 [00:00<00:00, 62.23it/s]


**Note on output_path:** The `output_path` argument to `generate_image_embeddings` is currently commented out due to an unresolved issue with the lab's shared embedding generator (tracked separately). Until fixed, embeddings will be written to `preprocessing/output/image_embeddings/clip_image_embeddings.csv` and **must be moved by hand** to `data/embeddings/clip_sketch_embeddings/` before running the cells below.

In [3]:
draw_embeddings = pd.read_csv("../data/embeddings/clip_sketch_embeddings/image_embeddings/clip_image_embeddings_csv.csv")

Here, we choose to column-wize z-score out embeddings so no single feature (out of 512-dimensions) overweights another for downstream analysis

In [4]:
embedding_cols = [str(i) for i in range(512)]  # or draw_embeddings.columns[:512]

# column-wise z-scoring
draw_embeddings[embedding_cols] = (draw_embeddings[embedding_cols] - draw_embeddings[embedding_cols].mean()) / draw_embeddings[embedding_cols].std()


In [5]:
draw_embeddings["embedding"] = draw_embeddings[embedding_cols].values.tolist()

/tmp/ipykernel_3181519/1679432702.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  draw_embeddings["embedding"] = draw_embeddings[embedding_cols].values.tolist()


In [6]:
# drop original 0–511 columns
draw_embeddings = draw_embeddings.drop(columns=embedding_cols)

In [7]:
draw_embeddings['embedding'][0]

[-0.5459077773548129,
 0.95730626101833,
 -1.9158020559173168,
 -0.46761182852998123,
 -2.148850516669877,
 0.7626254806715254,
 -1.825867674575075,
 1.2044898225746037,
 -1.4635526427081962,
 1.2979998426641757,
 0.7491787968707225,
 -0.33484366019221684,
 1.1209976222767373,
 -1.6158747488800236,
 0.08823531826901838,
 -0.5156846565909233,
 1.2158395206335266,
 -0.32486617311824517,
 -0.25389544559073557,
 1.2447611317172607,
 0.6395416695163219,
 -0.623547981420152,
 -0.2517412268855506,
 -1.1001930354621559,
 2.1759243315652737,
 0.7950972345625422,
 0.0726597717931766,
 0.48016551475114244,
 -0.4228178145436766,
 -0.9537642152809529,
 -0.30299029322203463,
 -2.126908034943927,
 -0.299412074757803,
 0.5241610093238177,
 0.43746263774758454,
 -0.42261121199477053,
 0.5330789712039218,
 1.1925728273587184,
 0.9543051672226442,
 -1.1111687887091084,
 1.5292564262044681,
 1.7592361467521258,
 0.31052461742773046,
 -0.3200210138721633,
 -1.2005299048626557,
 -0.3284452969756872,
 0.6014

Add back metadata

In [8]:
metadata_df = pd.read_csv("../data/clean/drawing_highres_clean.csv")

In [9]:
metadata_df.head()

,session_id,trial_num,text1,participant_id,image1,submit_time,submit_date,submit_date_readable,start_time,trial_duration,num_strokes,draw_duration,mean_intensity,bounding_box,age_months,age_yrs,row_id_no
0,cdm_run_v81752869472107,6,a_seahorse,AD001,/labs/vislearnlab/experiments/birch-draw/data/...,1.752870e+12,2025-07-18T20:14:10.295Z,2025-07-18 13:14:10.295000,1.752870e+12,10.183,4,8.058,0.008371,"(np.int64(52), np.int64(142), np.int64(77), np...",229.0,19.13,/labs/vislearnlab/experiments/birch-draw/data/...
1,cdm_run_v81752869472107,9,an_octopus,AD001,/labs/vislearnlab/experiments/birch-draw/data/...,1.752870e+12,2025-07-18T20:16:31.074Z,2025-07-18 13:16:31.074000,1.752870e+12,15.795,2,14.056,0.023079,"(np.int64(62), np.int64(198), np.int64(28), np...",229.0,19.13,/labs/vislearnlab/experiments/birch-draw/data/...
2,cdm_run_v81752869472107,5,a_penguin,AD001,/labs/vislearnlab/experiments/birch-draw/data/...,1.752870e+12,2025-07-18T20:13:25.570Z,2025-07-18 13:13:25.570000,1.752870e+12,8.342,9,6.272,0.021345,"(np.int64(32), np.int64(210), np.int64(55), np...",229.0,19.13,/labs/vislearnlab/experiments/birch-draw/data/...
3,cdm_run_v81752869472107,4,this_square,AD001,/labs/vislearnlab/experiments/birch-draw/data/...,1.752870e+12,2025-07-18T20:12:49.015Z,2025-07-18 13:12:49.015000,1.752870e+12,6.462,1,3.374,0.011858,"(np.int64(59), np.int64(164), np.int64(57), np...",229.0,19.13,/labs/vislearnlab/experiments/birch-draw/data/...
4,cdm_run_v81752869472107,3,shape,AD001,/labs/vislearnlab/experiments/birch-draw/data/...,1.752870e+12,2025-07-18T20:12:39.029Z,2025-07-18 13:12:39.029000,1.752870e+12,12.635,1,10.317,0.014728,"(np.int64(1), np.int64(218), np.int64(25), np....",229.0,19.13,/labs/vislearnlab/experiments/birch-draw/data/...


In [10]:
# Rename generated columns and clean target category
draw_embeddings = draw_embeddings.rename(columns={"text": "target_category"})
draw_embeddings["target_category"] = draw_embeddings["target_category"].str.replace(r"^.*?_", "", regex=True)

# Merge metadata — drawing_highres_clean uses image1 as the path key
draw_emb = draw_embeddings.merge(
    metadata_df[["image1", "participant_id", "age_months"]],
    left_on="row_id",
    right_on="image1",
    how="left"
).drop(columns="image1")

In [11]:
draw_emb.head()

,row_id,target_category,embedding,participant_id,age_months
0,/labs/vislearnlab/experiments/birch-draw/data/...,seahorse,"[-0.5459077773548129, 0.95730626101833, -1.915...",AD001,229.0
1,/labs/vislearnlab/experiments/birch-draw/data/...,octopus,"[-0.48118503700169896, -0.2941142848098647, -0...",AD001,229.0
2,/labs/vislearnlab/experiments/birch-draw/data/...,penguin,"[-0.8728405940102864, 0.30478917390415117, 1.0...",AD001,229.0
3,/labs/vislearnlab/experiments/birch-draw/data/...,square,"[-0.7671821033483651, -0.6118731067913185, -0....",AD001,229.0
4,/labs/vislearnlab/experiments/birch-draw/data/...,shape,"[-0.5719075106590553, -1.2029306686662862, -0....",AD001,229.0


In [12]:
sea_animals = ["crab", "octopus", "seahorse", "shark", "whale", "turtle"]

draw_emb_meta = draw_emb[
    draw_emb["target_category"].isin(sea_animals)
]

In [13]:
draw_emb_meta.head()

,row_id,target_category,embedding,participant_id,age_months
0,/labs/vislearnlab/experiments/birch-draw/data/...,seahorse,"[-0.5459077773548129, 0.95730626101833, -1.915...",AD001,229.0
1,/labs/vislearnlab/experiments/birch-draw/data/...,octopus,"[-0.48118503700169896, -0.2941142848098647, -0...",AD001,229.0
5,/labs/vislearnlab/experiments/birch-draw/data/...,crab,"[-0.023700368010029844, 0.08183493872786367, 0...",AD001,229.0
6,/labs/vislearnlab/experiments/birch-draw/data/...,turtle,"[0.15110634952487648, 0.06575085020781475, 0.0...",AD001,229.0
7,/labs/vislearnlab/experiments/birch-draw/data/...,shark,"[0.7529725162273385, -1.3186576470422475, 0.61...",AD001,229.0


In [15]:
draw_emb_meta.to_csv(
    "../data/embeddings/clip_sketch_embeddings/image_embeddings/draw_embeddings_with_metadata.csv",
    index=False
)